# 分类数据分箱与编码

学习目标：用固定类别表达业务顺序，按阈值或分位数分箱，并生成列集合一致的分类编码。

前置知识：Series、数据类型转换、排序、频数统计、缺失值与区间端点。

运行环境：Python 3.12、pandas 3.0；示例按 pandas 3.0.6 编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制数据，后续单元沿用已导入的 pd。类别顺序和分箱阈值是本例约定。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 按业务顺序排列记录

优先级“低、中、高”有业务顺序，普通字符串的排序规则不能替代这个约定。CategoricalDtype 描述类别集合及是否有序；将一列转换为这个类型，再排序，就可以使用指定的类别顺序。

下面先按优先级从低到高排列三个任务。categories 的排列定义顺序，ordered=True 允许按这个顺序进行大小比较。

In [1]:
import pandas as pd

priority_type = pd.CategoricalDtype(["低", "中", "高"], ordered=True)
tasks = pd.DataFrame({"task": ["检查", "归档", "回复"], "priority": ["高", "低", "中"]})
tasks["priority"] = tasks["priority"].astype(priority_type)
ordered = tasks.sort_values("priority")
print(ordered)  # 归档、回复、检查依次对应低、中、高；原行标签跟随记录。
print(ordered.index.tolist(), ordered.shape)  # [1, 2, 0] (3, 2)。
print(tasks.dtypes)  # task 为 str，priority 为 category。

  task priority
1   归档        低
2   回复        中
0   检查        高
[1, 2, 0] (3, 2)
task             str
priority    category
dtype: object


## 2 类别、编码与缺失

Categorical 是保存分类值的数组；放入 Series 后，可通过 cat 访问类别信息。类别必须唯一且不含缺失值，观测值本身可以缺失。仅写 dtype="category" 会从输入推断类别；需要跨批次一致时，应显式给出 CategoricalDtype。

codes 表示每个值在 categories 中的位置，从 0 开始；缺失用 −1。编码用于查找类别，不是连续测量值，不能据此计算“平均优先级”。

In [2]:
levels = pd.Series(pd.Categorical(["高", "低", None], dtype=priority_type), name="priority")
print(levels.cat.categories.tolist(), levels.cat.ordered)  # ['低', '中', '高'] True。
print(levels.cat.codes.tolist())  # [2, 0, -1]；中未出现，但仍属于允许的类别。
print(levels.isna().tolist())  # [False, False, True]。
print(levels.value_counts(sort=False, dropna=False))  # 低、高和缺失各 1，中为 0。

['低', '中', '高'] True
[2, 0, -1]
[False, False, True]
priority
低      1
中      0
高      1
NaN    1
Name: count, dtype: int64


## 3 未见类别与比较条件

### 3.1 转换与赋值的区别

当前版本直接转换固定分类类型时，不在类别集合中的值会变成缺失，并发出 Pandas4Warning：这种行为已弃用，未来会报错。下面仅作为边界演示，局部捕获并检查警告。实际处理应先检查成员关系，再明确拒绝或映射未知值，不能依赖这个兼容行为。

保留输入才能区分原本缺失与未知类别。给已有分类 Series 直接赋一个新类别则会失败，需要先明确是否扩充类别集合。

In [3]:
import warnings

raw = pd.Series(["低", "紧急", None], name="raw")
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always", pd.errors.Pandas4Warning)
    converted = raw.astype(priority_type)
assert len(captured) == 1 and captured[0].category is pd.errors.Pandas4Warning
print(captured[0].category.__name__)  # Pandas4Warning：未知类别直接转换的兼容行为已弃用。
unknown = raw.notna() & converted.isna()
print(raw[unknown])  # 行 1 的“紧急”需要单独处理，行 2 的原始缺失不算未知类别。
print(converted.cat.codes.tolist())  # [0, -1, -1]；转换后已不能仅凭编码区分两种原因。

# 预期 TypeError：“紧急”不在现有类别集合中，不能直接赋给分类值。
converted.loc[0] = "紧急"

Pandas4Warning
1    紧急
Name: raw, dtype: str
[0, -1, -1]


TypeError: Cannot setitem on a Categorical with a new category (紧急), set the categories first

### 3.2 有序与无序

无序分类可以判断相等，但不能进行大小比较。有序分类与标量比较时使用类别顺序；两个分类 Series 比较大小时，还需要类别集合及其顺序一致，Series 本身的标签也要符合比较要求。

下面沿用 levels。缺失位置的大小比较为 False；若任务要保留“未知”的标记，应另外保留 isna 结果。

In [4]:
print((levels > "低").tolist())  # [True, False, False]。
unordered = levels.cat.as_unordered()
print((unordered == "低").tolist())  # [False, True, False]。

# 预期 TypeError：unordered 是无序分类，不能进行大于比较。
unordered > "低"

[True, False, False]
[False, True, False]


TypeError: Unordered Categoricals can only compare equality or not

In [5]:
different = levels.cat.reorder_categories(["高", "中", "低"])

# 预期 TypeError：两个有序分类的类别顺序不同，不能逐项比较大小。
levels > different

TypeError: Categoricals can only be compared if 'categories' are the same.

## 4 管理类别

### 4.1 更名与增加

rename_categories 修改类别名称；add_categories 增加允许的类别，但不会自动产生对应观测。它们返回新结果，需要接住返回值。下面继续使用 levels，将名称改成英文，再单独展示新增“紧急”后的赋值。

In [6]:
renamed = levels.cat.rename_categories({"低": "low", "中": "medium", "高": "high"})
expanded = levels.cat.add_categories(["紧急"])
expanded.loc[2] = "紧急"
print(renamed.tolist())  # high、low 和缺失；类别顺序保持对应关系。
print(expanded.cat.categories.tolist())  # ['低', '中', '高', '紧急']，新类别追加在末尾。
print(expanded.tolist())  # 高、低、紧急。
print(levels.isna().tolist())  # 原 Series 仍为 [False, False, True]。

['high', 'low', nan]
['低', '中', '高', '紧急']
['高', '低', '紧急']
[False, False, True]


### 4.2 删除、重排与整体设置

remove_categories 删除允许的类别，原来属于该类别的值会变成缺失；remove_unused_categories 只清除本批数据未出现的类别。固定跨批次类别时，不应随意删除未使用类别。

reorder_categories 要求新旧类别集合完全一致，只调整排列；set_categories 可以同时增加、删除和重排，遗漏原类别也会使对应值变成缺失。下面沿用 levels，对照几种操作的目的。

In [7]:
print(levels.cat.remove_categories(["高"]).cat.codes.tolist())  # [-1, 0, -1]。
print(levels.cat.remove_unused_categories().cat.categories.tolist())  # ['低', '高']。
reordered = levels.cat.reorder_categories(["高", "中", "低"])
print(reordered.sort_values().tolist())  # 高、低、缺失；改顺序没有改类别名称。
restricted = levels.cat.set_categories(["低", "中"])
print(restricted.isna().tolist())  # [True, False, True]，被移除的高也变成缺失。

[-1, 0, -1]
['低', '高']
['高', '低', nan]
[True, False, True]


## 5 按阈值分箱

cut 把一维数值分到区间，称为分箱。bins 给出递增边界，labels 给每个区间命名，数量应等于区间数。默认 right=True 表示右闭，例如 (0, 10] 不含 0、包含 10；include_lowest=True 另外包含最左端点。

下面时长单位为分钟，边界是 0、10、20。决定“恰好 10 分钟”属于哪组之前，先看端点是实心还是空心。

![两条分箱数轴比较右闭且包含最左端点与左闭右开，展示 0、10、20 的归属。](image/illustration/10-01-bin-endpoints.svg)

区间在图中略微上下错开，只为露出同一边界处的两个端点，不表示测量值有两个维度。右闭示例把 10 归为“短”，左闭示例把 10 归为“长”。

下面保留原始 minutes 与分类结果并排查看，尤其核对 0、10、20。超出范围与原始缺失都会得到缺失，不能仅从 band 的 NaN 推断原因。

In [8]:
minutes = pd.Series([-1.0, 0.0, 10.0, 20.0, 21.0, None])
bands = pd.cut(minutes, bins=[0, 10, 20], labels=["短", "长"], include_lowest=True)
print(pd.DataFrame({"minutes": minutes, "band": bands}))
# 0、10 属于短；20 属于长；-1、21 和原始缺失得到 NaN。
print(bands.dtype, bands.cat.ordered)  # category True。
left_closed = pd.cut(minutes, bins=[0, 10, 20], right=False, labels=["短", "长"])
print(left_closed.tolist())  # 左闭右开：0 属于短，10 属于长，20 在范围外。

   minutes band
0     -1.0  NaN
1      0.0    短
2     10.0    短
3     20.0    长
4     21.0  NaN
5      NaN  NaN
category True
[nan, '短', '长', nan, nan, nan]


## 6 等宽与分位数分箱

### 6.1 cut 与 qcut

给 cut 一个整数 bins，会根据当前输入范围建立近似等宽区间，并调整范围以包含端点。qcut 按样本分位数选择边界，目标是让每箱记录数接近，而不是让区间宽度相同。重复值等条件会影响这个目标。

下面使用偏斜的六个数值；retbins=True 同时返回实际边界。业务阈值应固定使用明确的 bins，不应每个新批次都重新计算。

In [9]:
values = pd.Series([0, 1, 2, 3, 4, 100])
width_bins, width_edges = pd.cut(values, bins=2, retbins=True)
quantile_bins, quantile_edges = pd.qcut(values, q=2, retbins=True)
print(width_edges, width_bins.value_counts(sort=False).tolist())  # [-0.1, 50, 100]，数量 [5, 1]。
print(quantile_edges, quantile_bins.value_counts(sort=False).tolist())  # [0, 2.5, 100]，数量 [3, 3]。
print(quantile_bins.cat.categories)  # 区间标签的显示精度不代替 retbins 返回的边界。

[ -0.1  50.  100. ] [5, 1]
[  0.    2.5 100. ] [3, 3]
IntervalIndex([(-0.001, 2.5], (2.5, 100.0]], dtype='interval[float64, right]')


### 6.2 重复边界与跨批次使用

大量重复值可能让 qcut 的多个分位点重合。默认会报错；duplicates="drop" 删除重复边界后，实际箱数可能少于请求数量，也不能保证各箱等频。

若已有一批数据确定的分位数边界，要在另一批复用，应把返回的边界交给 cut，并说明端点规则和超出范围的处理。

In [10]:
repeated = pd.Series([0, 0, 0, 0, 1, 1])

# 预期 ValueError：重复值使四分位分箱的边界重复，默认不允许相同箱边界。
pd.qcut(repeated, q=4)

ValueError: Bin edges must be unique: Index([0.0, 0.0, 0.0, 0.75, 1.0], dtype='float64').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [11]:
reduced = pd.qcut(repeated, q=4, duplicates="drop")
print(len(reduced.cat.categories), reduced.value_counts(sort=False).tolist())  # 2 个箱，数量 [4, 2]。
later = pd.Series([0.0, 3.0, 101.0])
reused = pd.cut(later, bins=quantile_edges, include_lowest=True)
print(reused.cat.codes.tolist())  # [0, 1, -1]，101 超出了建立规则时的最大边界。

2 [4, 2]
[0, 1, -1]


## 7 指示变量与稳定列集合

### 7.1 get_dummies

get_dummies 为类别生成指示变量（dummy variables），每列表示是否属于某个类别，默认值类型为 bool。下面显式使用 int8 展示 0 和 1。类别编码与指示变量不同：前者是一列位置编号，后者是多列成员标记。

对 DataFrame 可用 columns 指定要编码的列，其他列保留。dummy_na=True 为缺失生成一列；默认 False 时，缺失行在这些指示列中全部为 0。

In [12]:
colors_type = pd.CategoricalDtype(["red", "blue", "green"])
items = pd.DataFrame({"quantity": [1, 2, 3],
                      "color": pd.Series(["red", "blue", None], dtype=colors_type)})
encoded = pd.get_dummies(items, columns=["color"], dummy_na=True, dtype="int8")
print(encoded)  # quantity 保留；green 列即使未出现仍存在；最后一行 color_nan 为 1。
print(encoded.columns.tolist())  # quantity、color_red、color_blue、color_green、color_nan。
print(encoded.dtypes)  # quantity 为 int64，指示列为 int8。

   quantity  color_red  color_blue  color_green  color_nan
0         1          1           0            0          0
1         2          0           1            0          0
2         3          0           0            0          1
['quantity', 'color_red', 'color_blue', 'color_green', 'color_nan']
quantity       int64
color_red       int8
color_blue      int8
color_green     int8
color_nan       int8
dtype: object


### 7.2 减少一列的含义

drop_first=True 去掉首个类别对应的列。若同时使用默认 dummy_na=False，第一个类别和缺失都可能变成全零，无法仅凭输出区分。

下面继续用 items 的颜色列，先观察信息丢失，再考虑是否适合当前任务。

In [13]:
reduced_encoding = pd.get_dummies(items["color"], drop_first=True, dtype="int8")
print(reduced_encoding)  # red 行与缺失行均为全零；blue 行的 blue 为 1。
print(reduced_encoding.loc[0].equals(reduced_encoding.loc[2]))  # True，原始含义却不同。

   blue  green
0     0      0
1     1      0
2     0      0
True


### 7.3 不同批次沿用同一规则

独立推断类别会使不同批次的编码列不一致。应先约定类别集合，并在转换前识别未知输入，再决定拒绝、记录为异常或映射到专门类别。

下面沿用 colors_type。本例先打印未知记录，再明确把它视为缺失参与展示；这是一项处理决定，不代表未知类别天然等于缺失。固定类型和相同编码参数使两个结果的列集合与顺序一致。

In [14]:
first = pd.Series(["red", "blue"], dtype=colors_type)
new_raw = pd.Series(["blue", "yellow", None])
unknown_color = new_raw.notna() & ~new_raw.isin(colors_type.categories)
print(new_raw[unknown_color])  # 行 1 的 yellow 是未知类别，先保留这条错误记录。
accepted = new_raw.mask(unknown_color).astype(colors_type)
first_codes = pd.get_dummies(first, dummy_na=True, dtype="int8")
new_codes = pd.get_dummies(accepted, dummy_na=True, dtype="int8")
print(first_codes.columns.tolist())  # red、blue、green 和 NaN 指示列。
print(new_codes)  # blue 正常编码；未知记录与原始缺失均进入缺失指示列。
print(first_codes.columns.equals(new_codes.columns))  # True。
assert first_codes.columns.equals(new_codes.columns)

1    yellow
dtype: str
['red', 'blue', 'green', nan]
   red  blue  green  NaN
0    0     1      0    0
1    0     0      0    1
2    0     0      0    1
True


## 8 选学：区间索引与紧凑编码
### 8.1 IntervalIndex

IntervalIndex 保存一组区间，from_breaks 根据边界创建相邻区间，closed 决定包含哪侧端点。把它交给 cut 时，区间自身决定边界，right 和 labels 参数不再用于重新定义这些区间。

适合已经明确持有一组区间规则、希望直接复用的情况；用于分箱的区间不能重叠。

In [15]:
intervals = pd.IntervalIndex.from_breaks([0, 10, 20], closed="left")
result = pd.cut(pd.Series([0, 10, 20]), bins=intervals)
print(intervals)  # [0, 10)、[10, 20)，均为左闭右开。
print(result.cat.codes.tolist())  # [0, 1, -1]，20 不属于任何区间。

IntervalIndex([[0, 10), [10, 20)], dtype='interval[int64, left]')
[0, 1, -1]


### 8.2 factorize

factorize 同时返回整数编码和唯一值集合，默认 sort=False 按首次出现顺序编号；sort=True 会排序唯一值并相应调整编码。它适合一次性识别重复值，不自动提供跨批次稳定字典。

默认缺失码为 −1。恢复时不能直接拿 −1 索引唯一值数组，否则会误取最后一个类别；先把有效位置分开。

In [16]:
labels = pd.Series(["blue", "red", None, "blue"])
codes, uniques = pd.factorize(labels, sort=False)
print(codes.tolist(), uniques.tolist())  # [0, 1, -1, 0] 与 ['blue', 'red']。
restored = pd.Series(pd.NA, index=labels.index, dtype="string")
valid = codes >= 0
restored.loc[valid] = uniques.take(codes[valid]).to_numpy()
print(restored.tolist())  # blue、red、<NA>、blue，缺失未被误恢复成 red。
other_codes, other_uniques = pd.factorize(pd.Series(["red", "blue"]))
print(other_codes.tolist(), other_uniques.tolist())  # 本批 0 代表 red，与上面的字典不同。

[0, 1, -1, 0] ['blue', 'red']
['blue', 'red', <NA>, 'blue']
[0, 1] ['red', 'blue']


## 9 选学：从指示变量恢复标签
from_dummies 可以把完整的指示列还原为标签表，sep 指定列名前缀与类别之间的分隔符。它不会自动恢复原先的有序分类规则，返回后仍需检查 dtype 和类别约定。

同一变量的一行出现多个 1 会失败；全零行没有明确类别时也会失败。default_category 可以为全零行规定类别，但不能凭空辨别曾被合并的原始缺失与基准类别。

In [17]:
flags = pd.DataFrame({"color_red": [1, 0], "color_blue": [0, 1]}, dtype="int8")
decoded = pd.from_dummies(flags, sep="_")
print(decoded)  # 两行一列 color，标签为 red、blue。
print(decoded.dtypes)  # 本环境为 str，并不是带固定顺序的 category。
all_zero = pd.DataFrame({"color_red": [0], "color_blue": [0]}, dtype="int8")

# 预期 ValueError：该行的虚拟变量全为零，且没有提供默认类别，无法还原标签。
pd.from_dummies(all_zero, sep="_")

  color
0   red
1  blue
color    str
dtype: object


ValueError: Dummy DataFrame contains unassigned value(s); First instance in row: 0

In [18]:
print(pd.from_dummies(all_zero, sep="_", default_category="unknown"))  # 显式指定 unknown。

     color
0  unknown

## 本章小结

（1）CategoricalDtype 固定类别及顺序；类别编码只是位置，缺失码为 −1。

（2）转换前保存未知类别记录。更名、增加、删除与重排各有不同目的，删除类别可能产生缺失。

（3）cut 适合固定阈值或等宽分箱，qcut 按样本分位数分箱；检查端点、重复边界和范围外输入。

（4）固定类别集合与编码参数可保持跨批次列一致。缺失、未知类别和被删除的首个指示列可能影响可恢复的信息。

## 练习

（1）下面先将未知类别显式映射为缺失，再转换类型。先预测分类编码和缺失位置，再运行。为什么未知类别与原始缺失会得到相同编码？应保留哪项信息才能区分？

In [19]:
exercise_type = pd.CategoricalDtype(["A", "B", "C"], ordered=True)
exercise_raw = pd.Series(["C", None, "D", "A"])
exercise_unknown = exercise_raw.notna() & ~exercise_raw.isin(exercise_type.categories)
exercise = exercise_raw.mask(exercise_unknown).astype(exercise_type)
print(exercise.cat.codes.tolist())
print(exercise.isna().tolist())
# 先记录预测，再结合原输入解释；不要仅凭转换结果推断缺失原因。

[2, -1, -1, 0]
[False, True, True, False]


（2）将 0 至 60 分钟分为“短”和“长”：0 至 30 分钟含两端，超过 30 至 60 分钟含右端。完成分箱，再解释为什么新一天的数据仍应使用相同边界。若任务改为把本批记录尽量分成数量接近的两组，应选择什么方法？

In [20]:
durations = pd.Series([0, 10, 30, 31, 60, 61])
# 在此选择 cut 参数并打印结果；前 3 项为短，随后 2 项为长，61 为缺失。
# 分别解释固定业务阈值与本批分位数的适用场合，不把两种目标混用。

（3）对两个批次生成列顺序相同的颜色指示表，允许 red、blue、green，并单独标记缺失。保留第二批的未知值记录，再将未知值按本题约定映射为缺失。解释为什么不能只比较两表的列数。

In [21]:
batch_a = pd.Series(["red", "green"])
batch_b = pd.Series(["blue", "yellow", None])
# 在此定义固定分类类型，识别未知值，然后编码。
# 检查：未知值仅 yellow；列的标签和顺序一致，数量为 4，指示值类型一致。
# 核对每行所属类别以及未知记录与原始缺失各自的来源。

（4）为优先级增加“紧急”，然后按“紧急、高、中、低”排序。说明增加类别与更改排序各使用什么方法；若仅将“低”改名为“普通”，应如何避免把原值变成缺失？

In [22]:
priorities = pd.Series(["低", "高", "中"], dtype=priority_type)
# 在此增加并重排类别，再单独演示“低”改名为“普通”。
# 检查：增加未使用类别不改变行数；重排不丢值；更名后的缺失数量仍为 0。

### 重点练习提示（第 3 题）

提示一：先固定类别全集和顺序，再识别未知值；不要让每批数据自行推断编码列。

提示二：转换前保存非缺失但不在类别中的记录；对两批使用相同 CategoricalDtype，并启用 dummy_na。

### 参考解析（第 3 题）

固定类别顺序为 red、blue、green。batch_b 中 yellow 是未知值，None 是原有缺失，须在转换前分别保留标记；按题约定把未知值映射为缺失后再转换。两批均用 get_dummies(..., dummy_na=True, dtype=bool)，得到相同顺序的四列：red、blue、green、缺失。A 批两行分别点亮 red 与 green；B 批三行分别点亮 blue、缺失、缺失。最后两行编码相同不代表来源相同；只比较列数也不能发现列名或列序错位。

## 参考与引用来源

本章新增示意图由 CMYK Labs 原创，依据下表对应概念与本章教学输入绘制；示意图不作为实际运行截图或数学证明。

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [Categorical](https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html) 的 categories、ordered、codes、未知值与缺失；[Categorical data](https://pandas.pydata.org/docs/user_guide/categorical.html) 的 CategoricalDtype、Sorting and order、Comparisons、Missing data；类别管理 API：[rename_categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.rename_categories.html)、[add_categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.add_categories.html)、[remove_categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.remove_categories.html)、[remove_unused_categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.remove_unused_categories.html)、[reorder_categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.reorder_categories.html)、[set_categories](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.set_categories.html)、[as_unordered](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.as_unordered.html) 的参数和返回值，并核对当前安装版本官方 docstring；[cut](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) 的 bins、right、labels、include_lowest、retbins 和 Notes；[qcut](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html) 的 q、duplicates、retbins；[get_dummies](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html) 的 columns、dtype、dummy_na、drop_first；[IntervalIndex.from_breaks](https://pandas.pydata.org/docs/reference/api/pandas.IntervalIndex.from_breaks.html) 的 closed，结合当前安装版本官方 docstring；[factorize](https://pandas.pydata.org/docs/reference/api/pandas.factorize.html) 的 sort、use_na_sentinel 与缺失示例；[from_dummies](https://pandas.pydata.org/docs/reference/api/pandas.from_dummies.html) 的 sep、default_category 与 Raises；[pandas 3.0.0](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html#other-deprecations) 的 Other Deprecations（GH 40996）：转换时含未登记的非缺失类别已弃用。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[categorical](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/categorical.rst)、[v3.0.0](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v3.0.0.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |